# 📰 A Supervisor Workflow with Human Review

**Pattern:** the supervisor architecture — see Part 2 of `03_five_multi_agent_architectures.ipynb` for its
origin and trade-offs.

A news agency receives short transfer rumours ("Lionel Messi will join Real Madrid in 2025") and must turn them
into publishable articles: on-topic, mentioning the player's **current club** and **market value**, and at least
**100 words** long. A supervisor — the "news chef" — checks the article and routes it to whichever specialist
agent can fix what is missing, until it passes. A human reviews the result before it is published.

## Learning Objectives
In this notebook, you will learn:
1. **Specialist sub-agents** - small LangGraph agents with their own tools and input/output schemas
2. **A grader** - structured output for judgement calls, plain code for what code can check
3. **A supervisor loop** - route to the specialist that fixes the first failing check, with a pass cap
4. **Human in the loop** - pause after the workflow, inspect or edit the answer, then resume

## Where this fits
`03_five_multi_agent_architectures.ipynb` sketched the supervisor with hard-coded strings. This is the same
shape with real agents, a real grader and a human checkpoint. Module 06 goes deeper into supervisors.

## Prerequisites
- `03_five_multi_agent_architectures.ipynb` and `04_agent_communication.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library deprecation warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the specialists, the grader and the text writer
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧑‍🔬 Part 1: Specialist Sub-Agents

Each specialist is its own small graph with an **input schema** (`article`), an **output schema**
(`agent_output`) and a private working state. The two researchers have a tool backed by a tiny fake database;
the text writer just expands text. A researcher's first model call is forced to call its tool (`tool_choice`),
so it can't skip the lookup and answer "not available" from its own guess.

In [ ]:
# ============================================================================
# PART 1A: THE CURRENT-CLUB RESEARCHER
# ============================================================================
from operator import add
from typing import Annotated, List, Literal, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode


class InputState(TypedDict):
    article: str


class OutputState(TypedDict):
    agent_output: str


class OverallState(InputState, OutputState):
    messages: Annotated[List[BaseMessage], add]


def build_researcher(tool_fn, instructions: str):
    """A one-tool agent: ask the model, run the tool if it asks, repeat until it answers."""
    first_step = llm.bind_tools([tool_fn], tool_choice=tool_fn.name)  # step 1 always looks it up
    model = llm.bind_tools([tool_fn])  # later steps may answer

    def call_model(state: OverallState) -> dict:
        messages = state.get("messages") or [HumanMessage(content=state["article"])]
        step_model = model if state.get("messages") else first_step
        response = step_model.invoke([SystemMessage(content=instructions)] + messages)
        new = [] if state.get("messages") else messages  # add the opening message on the first call
        return {"messages": new + [response], "agent_output": response.content}

    def should_continue(state: OverallState) -> Literal["tools", "__end__"]:
        return "tools" if getattr(state["messages"][-1], "tool_calls", None) else END

    graph = StateGraph(state_schema=OverallState, input_schema=InputState, output_schema=OutputState)
    graph.add_node("call_model", call_model)
    graph.add_node("tools", ToolNode([tool_fn]))
    graph.add_edge(START, "call_model")
    graph.add_conditional_edges("call_model", should_continue)
    graph.add_edge("tools", "call_model")
    return graph.compile()


@tool
def get_current_club(player_name: str):
    """Gets the current club of a player."""
    fake_db = {"Lionel Messi": "Inter Miami", "Cristiano Ronaldo": "Al Nassr FC"}
    return fake_db.get(player_name, "Current club information not available.")


current_club_researcher_agent = build_researcher(
    get_current_club,
    "You determine the current club of the player in the article. Use the tool. Answer with one short sentence "
    "stating the current club, or 'Current club information not available.'",
)

for article in ["Lionel Messi will join Real Madrid in 2025", "Thomas Müller will join Real Madrid in 2025"]:
    print(f"{article!r} -> {current_club_researcher_agent.invoke({'article': article})['agent_output']}")

In [ ]:
# ============================================================================
# PART 1B: THE MARKET-VALUE RESEARCHER AND THE TEXT WRITER
# ============================================================================
@tool
def get_market_value(player_name: str):
    """Gets the current market value of a player."""
    fake_market_value_db = {"Lionel Messi": "€50 million", "Cristiano Ronaldo": "€30 million"}
    return fake_market_value_db.get(player_name, "Market value information not available.")


market_value_researcher_agent = build_researcher(
    get_market_value,
    "You determine the market value of the player in the article. Use the tool. Answer with one short sentence "
    "stating the market value, or 'Market value information not available.'",
)


class WriterState(InputState, OutputState):
    pass


def expand_text_to_100_words(state: WriterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="Expand the following text to at least 120 words. Keep every fact; add context, not new facts. "
                              "Reply with the article text only."),
        HumanMessage(content=state["article"]),
    ])
    return {"agent_output": response.content}


writer_graph = StateGraph(state_schema=WriterState, input_schema=InputState, output_schema=OutputState)
writer_graph.add_node("expand_text_to_100_words", expand_text_to_100_words)
writer_graph.add_edge(START, "expand_text_to_100_words")
writer_graph.add_edge("expand_text_to_100_words", END)
text_writer_agent = writer_graph.compile()

print(market_value_researcher_agent.invoke({"article": "Lionel Messi will join Real Madrid in 2025"})["agent_output"])

---
## ⚖️ Part 2: The Grader ("News Chef")

Three checks need judgement — is the article about football transfers, does it mention the market value, does
it mention the current club — so an LLM answers them with structured output. The fourth check, **at least 100
words**, is a count: code does it exactly, so the model is never asked.

In [ ]:
# ============================================================================
# PART 2: STRUCTURED GRADING + A WORD COUNT IN CODE
# ============================================================================
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class ArticlePostabilityGrader(BaseModel):
    """Binary checks for whether a transfer article can be published."""
    off_or_ontopic: str = Field(description="The article is about football transfers: 'yes' or 'no'")
    mentions_market_value: str = Field(description="The article mentions the player's market value: 'yes' or 'no'")
    mentions_current_club: str = Field(description="The article mentions the player's current club (or says it is unavailable): 'yes' or 'no'")


grader_prompt = ChatPromptTemplate.from_messages([
    ("system", "You grade news articles. Answer each field with 'yes' or 'no':\n"
               "- off_or_ontopic: is the article about football transfers?\n"
               "- mentions_market_value: does it state the player's market value (e.g. 'market value' or an amount like '€50 million')?\n"
               "- mentions_current_club: does it name the player's current club, or say that information is unavailable?"),
    ("human", "News Article:\n\n{article}"),
])
news_chef = grader_prompt | llm.with_structured_output(ArticlePostabilityGrader)


def meets_100_words(article: str) -> str:
    return "yes" if len(article.split()) >= 100 else "no"


for article in ["Lionel Messi will switch to Real Madrid in 2025", "Today in Munich will be 9°C"]:
    print(article, "->", news_chef.invoke({"article": article}).model_dump(), "| 100 words:", meets_100_words(article))

---
## 🔁 Part 3: The Supervisor Loop

After every specialist, control returns to the news chef, which re-grades the *updated* article and routes to
the first failing check: market value → current club → length. Off-topic articles end immediately. A pass
counter caps the loop, so a grader that keeps saying "no" cannot spin forever.

When every check passes, the news chef copies the current article into `final_article`. That is the only
place it is written, so `final_article` is always the exact version that passed — whether or not any
specialist ran.

In [ ]:
# ============================================================================
# PART 3A: THE WORKFLOW GRAPH
# ============================================================================
MAX_PASSES = 5


class InputArticleState(TypedDict):
    article: str


class OutputFinalArticleState(TypedDict):
    final_article: str
    off_or_ontopic: str


class SharedArticleState(InputArticleState, OutputFinalArticleState):
    mentions_market_value: str
    mentions_current_club: str
    meets_100_words: str
    passes: int


CHECKS = ("off_or_ontopic", "mentions_market_value", "mentions_current_club", "meets_100_words")


def news_chef_node(state: SharedArticleState) -> dict:
    grade = news_chef.invoke({"article": state["article"]})
    update = {**grade.model_dump(), "meets_100_words": meets_100_words(state["article"]),
              "passes": state.get("passes", 0) + 1}
    print(f"  [news chef, pass {update['passes']}] { {k: v for k, v in update.items() if k != 'passes'} }")
    if all(update[check] == "yes" for check in CHECKS):
        update["final_article"] = state["article"]  # the version that passed every check, set nowhere else
    return update


def market_value_researcher_node(state: SharedArticleState) -> dict:
    found = market_value_researcher_agent.invoke({"article": state["article"]})["agent_output"]
    return {"article": f"{state['article']} {found}"}


def current_club_researcher_node(state: SharedArticleState) -> dict:
    found = current_club_researcher_agent.invoke({"article": state["article"]})["agent_output"]
    return {"article": f"{state['article']} {found}"}


def word_count_rewriter_node(state: SharedArticleState) -> dict:
    expanded = text_writer_agent.invoke({"article": state["article"]})["agent_output"]
    return {"article": expanded}  # the news chef re-grades it; only a full pass sets final_article


def news_chef_decider(state: SharedArticleState) -> Literal[
    "market_value_researcher", "current_club_researcher", "word_count_rewriter", "__end__"
]:
    if state["off_or_ontopic"] == "no" or state["passes"] >= MAX_PASSES:
        return END
    if state["mentions_market_value"] == "no":
        return "market_value_researcher"
    if state["mentions_current_club"] == "no":
        return "current_club_researcher"
    if state["meets_100_words"] == "no":
        return "word_count_rewriter"
    return END


workflow = StateGraph(state_schema=SharedArticleState, input_schema=InputArticleState, output_schema=OutputFinalArticleState)
workflow.add_node("news_chef", news_chef_node)
workflow.add_node("market_value_researcher", market_value_researcher_node)
workflow.add_node("current_club_researcher", current_club_researcher_node)
workflow.add_node("word_count_rewriter", word_count_rewriter_node)
workflow.add_edge(START, "news_chef")
workflow.add_conditional_edges("news_chef", news_chef_decider)
for specialist in ("market_value_researcher", "current_club_researcher", "word_count_rewriter"):
    workflow.add_edge(specialist, "news_chef")  # every specialist reports back to the supervisor

app = workflow.compile()
print("✅ Supervisor workflow compiled")

In [ ]:
# ============================================================================
# PART 3B: AN OFF-TOPIC ARTICLE AND A TRANSFER RUMOUR
# ============================================================================
print("--- Off-topic ---")
print(app.invoke({"article": "Today in Munich will be 9°C"}))

print("\n--- Transfer rumour ---")
result = app.invoke({"article": "Lionel Messi will join Real Madrid in 2025"})
final = result.get("final_article", "")
print(f"\nFinal article ({len(final.split())} words):\n{final or '(none)'}")
for label, ok in [("at least 100 words", len(final.split()) >= 100),
                  ("states the current club (Inter Miami)", "Inter Miami" in final),
                  ("states the market value (€50 million)", "€50 million" in final)]:
    print(f"{'✅' if ok else '❌'} final article {label}")

Follow the news chef's printed passes: the rumour should fail the market-value check first, then the club check,
then the length check, each time routed to the specialist that fixes it, and end once all checks pass (or at
`MAX_PASSES`). The grader is a model, so now and then it marks a check "no" twice and the same specialist runs
again — that is the loop doing its job, and the pass cap bounds it. The ✅ lines check the final article in code. The final word count is computed in code, so you can verify the length claim directly.

### 3.1 ✅ An article that passes on the first look

An article that already meets every check needs no specialist: the news chef grades it once, all four checks
pass, and the workflow ends. `final_article` must still be set — the publishing step in Part 4 reads it.

In [ ]:
# ============================================================================
# PART 3C: AN ARTICLE THAT PASSES EVERY CHECK WITHOUT A SPECIALIST
# ============================================================================
clean_article = (
    "Lionel Messi will join Real Madrid in 2025, according to sources close to the player. The Argentine forward "
    "currently plays for Inter Miami in Major League Soccer, where he moved in 2023 after two seasons at Paris "
    "Saint-Germain. His current market value is estimated at €50 million, a figure that reflects both his age and "
    "his continuing influence on the pitch. Real Madrid are said to see him as an experienced leader for a young "
    "squad, while Inter Miami would expect a fee close to that valuation before agreeing to any deal. Neither club "
    "has confirmed talks, and the player's representatives declined to comment on the reports this week."
)
print(f"Input: {len(clean_article.split())} words\n")

clean_result = app.invoke({"article": clean_article})
print(f"\n{'✅' if clean_result.get('final_article') == clean_article else '❌'} "
      "final_article is the input article, unchanged")

---
## 🙋 Part 4: Human in the Loop

The whole workflow becomes one node of an outer graph, followed by an `api_call_node` that "publishes". Compiling
with a checkpointer and `interrupt_after=["newsagent_node"]` pauses the run after the article is written, so a
human can inspect it, edit it with `update_state`, and resume with `invoke(None, ...)`.

> **Static vs. dynamic interrupts**: `interrupt_after=[...]` is *static* — fixed at compile time, it pauses after
> the named node on every run, and the human edits state from outside. The *dynamic* alternative is calling
> `interrupt(payload)` inside a node: the node decides at run time whether to pause and what to show the reviewer,
> and the run resumes with `Command(resume=...)`. See
> [`05_long_term_procedural_memory_sqlite.ipynb`](../../m24_long_term_agentic_memory/optional/02d_long_term_procedural_memory_sqlite.ipynb) (module 03),
> where a human approves a learned procedure, and
> [`04_hotel_edge_cases_and_errors.ipynb`](../../m27_multi_agent_builds_deep_agents/notebooks/02a_hotel_edge_cases_and_errors.ipynb) (module 10), where only
> requests that break a policy pause for review.

In [ ]:
# ============================================================================
# PART 4A: WRAP THE WORKFLOW, PAUSE BEFORE PUBLISHING
# ============================================================================
from langgraph.checkpoint.memory import InMemorySaver


class HumanInputState(TypedDict):
    question: str


class IntermediateState(HumanInputState):
    off_or_ontopic: str
    answer: str


class FinalState(IntermediateState):
    api_response: str
    status_code: int


def newsagent_node(state: IntermediateState) -> dict:
    response = app.invoke({"article": state["question"]})
    if "final_article" in response:
        answer = response["final_article"]
    elif response["off_or_ontopic"] == "no":
        answer = "Article not relevant for news agency"
    else:
        answer = f"Article did not pass its checks within {MAX_PASSES} passes"
    return {"answer": answer, "off_or_ontopic": response["off_or_ontopic"]}


def api_call_node(state: FinalState) -> dict:
    return {"status_code": 200, "api_response": f"API received answer: {state['answer']}"}


review_graph = StateGraph(FinalState, input_schema=HumanInputState, output_schema=FinalState)
review_graph.add_node("newsagent_node", newsagent_node)
review_graph.add_node("api_call_node", api_call_node)
review_graph.add_edge(START, "newsagent_node")
review_graph.add_edge("newsagent_node", "api_call_node")
review_graph.add_edge("api_call_node", END)

human_app = review_graph.compile(checkpointer=InMemorySaver(), interrupt_after=["newsagent_node"])
print("✅ Review graph compiled — pauses after newsagent_node")

In [ ]:
# ============================================================================
# PART 4B: RUN THREE THREADS UNTIL THE PAUSE
# ============================================================================
config_weather = {"configurable": {"thread_id": "weather"}}
config_messi = {"configurable": {"thread_id": "messi"}}
config_clean = {"configurable": {"thread_id": "clean"}}

human_app.invoke({"question": "The weather will be 9°C in Munich"}, config=config_weather)
human_app.invoke({"question": "Lionel Messi will join Real Madrid in 2025"}, config=config_messi)
human_app.invoke({"question": clean_article}, config=config_clean)

for name, cfg in [("weather", config_weather), ("messi", config_messi), ("clean", config_clean)]:
    snapshot = human_app.get_state(cfg)
    print(f"{name}: paused before {snapshot.next} — answer: {snapshot.values.get('answer', '')[:80]}")

In [ ]:
# ============================================================================
# PART 4C: APPROVE TWO, OVERRIDE ONE
# ============================================================================
# Weather: approve as-is by resuming
print(human_app.invoke(None, config=config_weather)["api_response"])

# Messi: a human overrides the answer before it is published
human_app.update_state(config_messi, {"answer": "Fake news!!!"})
print(human_app.invoke(None, config=config_messi)["api_response"])
print("messi next:", human_app.get_state(config_messi).next, "(empty = finished)")

# Clean article: approve as-is — the article that passed on the first look is what gets published
published = human_app.invoke(None, config=config_clean)["api_response"]
print(published[:120] + "...")
print(f"{'✅' if published == f'API received answer: {clean_article}' else '❌'} "
      "the clean article was published unchanged (not 'Article not relevant')")

The edited answer — not the model's — reached the API: `update_state` wrote the human's value into the paused
thread's state, and resuming continued from exactly that point. The clean article went through untouched:
it passed every check on the first look, so `final_article` held it and the API received it word for word.

---
## 📝 Summary

In this notebook, we learned:

### 1. Specialist sub-agents
- Small graphs with their own tools and input/output schemas, invoked as plain functions from a parent graph

### 2. A grader
- Structured output for judgement calls; a word count in code for what code can check exactly

### 3. A supervisor loop
- The supervisor re-grades after every specialist and routes to the first failing check, capped at `MAX_PASSES`
- Only a full pass sets `final_article`, so the published text is always the version that passed

### 4. Human in the loop
- `interrupt_after` + a checkpointer pause the run; `update_state` edits it; `invoke(None, ...)` resumes
- `interrupt_after` is static (every run pauses at that node); `interrupt()` inside a node pauses only when the node decides to

### Next Steps
- **Module 06 — `06_supervisor_and_hierarchical_agents`**: supervisors with typed routing decisions, and hierarchies of teams